# Preparando o ambiente

## Importação de Bibliotecas

In [1]:
#install
#!pip install -q -U duckdb matplotlib numpy pandas plotly scikit-learn scipy seaborn shap statsmodels xgboost 

In [1]:
#imports
import os
import time
import warnings
import pandas as pd
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from itertools import combinations
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor


## Obtenção das tabelas e dados

In [2]:
dataset_marketing = pd.read_csv('dados_marketing/dados_marketing.csv', sep=';')

# Análise Exploratória de Dados (EDA)

In [3]:
def eda_report(df, nome='DataFrame'):
    
    print(f"{'='*40}")
    print(f"EDA Report: {nome}")
    print(f"{'='*40}")
    
    # Colunas e Linhas
    print(f"\nColunas: {df.shape[1]}")
    print(f"Linhas:  {df.shape[0]}")
    
    # Dtypes
    print(f"\nDtypes:")
    print(f"  str:   {len(df.select_dtypes(include=['str','object']).columns)}")
    print(f"  int:   {len(df.select_dtypes(include='int').columns)}")
    print(f"  float: {len(df.select_dtypes(include='float').columns)}")
    print(f"  data:  {len(df.select_dtypes(include='datetime').columns)}")
    
    # Non-Null
    print(f"\nNon-Null:")
    for col in df.columns:
        print(f"  {col}: {df[col].notnull().sum()}")
    
    # Unique
    print(f"\nUnique:")
    for col in df.columns:
        print(f"  {col}: {df[col].nunique()}")
    
    # Duplicatas
    print(f"\nDados Duplicados: {df.duplicated().sum()}")
    print(f"{'='*40}")

In [4]:
dataset_marketing.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 27 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ID                              2000 non-null   int64  
 1   Ano Nascimento                  2000 non-null   int64  
 2   Escolaridade                    2000 non-null   str    
 3   Estado Civil                    2000 non-null   str    
 4   Salario Anual                   1981 non-null   float64
 5   Filhos em Casa                  2000 non-null   int64  
 6   Adolescentes em Casa            2000 non-null   int64  
 7   Data Cadastro                   2000 non-null   str    
 8   Dias Desde Ultima Compra        2000 non-null   int64  
 9   Gasto com Eletronicos           2000 non-null   int64  
 10  Gasto com Brinquedos            2000 non-null   int64  
 11  Gasto com Moveis                2000 non-null   int64  
 12  Gasto com Utilidades            2000 non-null

In [5]:
dataset_marketing.describe()

,ID,Ano Nascimento,Salario Anual,Filhos em Casa,Adolescentes em Casa,Dias Desde Ultima Compra,Gasto com Eletronicos,Gasto com Brinquedos,Gasto com Moveis,Gasto com Utilidades,...,Numero de Compras na Web,Numero de Compras via Catalogo,Numero de Compras na Loja,Numero Visitas WebSite Mes,Compra na Campanha 1,Compra na Campanha 2,Compra na Campanha 3,Compra na Campanha 4,Compra na Campanha 5,Comprou
count,2000.000000,2000.000000,1981.000000,2000.000000,2000.000000,2000.00000,2000.00000,2000.000000,2000.000000,2000.000000,...,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,5617.382500,1968.797000,52290.852600,0.447500,0.503000,43.73500,303.92850,26.316500,164.143500,37.587000,...,4.075000,2.635500,5.797500,5.327500,0.074000,0.073000,0.071500,0.066500,0.013000,0.160000
std,3259.910118,11.981468,25484.701911,0.535151,0.540497,25.85885,337.84483,40.317925,221.565768,54.748143,...,2.754663,2.885793,3.275952,2.440947,0.261836,0.260202,0.257723,0.249216,0.113302,0.366698
min,0.000000,1893.000000,1730.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2814.750000,1959.000000,35196.000000,0.000000,0.000000,22.00000,23.00000,1.000000,16.000000,3.000000,...,2.000000,0.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5492.000000,1970.000000,51766.000000,0.000000,0.000000,45.00000,175.50000,8.000000,67.000000,12.000000,...,4.000000,2.000000,5.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,8495.000000,1977.000000,68281.000000,1.000000,1.000000,66.00000,503.25000,32.000000,226.000000,50.000000,...,6.000000,4.000000,8.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,11191.000000,1996.000000,666666.000000,2.000000,2.000000,88.00000,1493.00000,199.000000,1725.000000,259.000000,...,27.000000,28.000000,13.000000,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [6]:
dataset_marketing.describe(include='str')

,Escolaridade,Estado Civil,Data Cadastro,Pais
count,2000,2000,2000,2000
unique,5,3,392,7
top,Curso Superior,Solteiro,5/04/2021,Estados Unidos
freq,1000,1197,32,977


In [7]:
dataset_marketing.isna().sum()

ID                                 0
Ano Nascimento                     0
Escolaridade                       0
Estado Civil                       0
Salario Anual                     19
Filhos em Casa                     0
Adolescentes em Casa               0
Data Cadastro                      0
Dias Desde Ultima Compra           0
Gasto com Eletronicos              0
Gasto com Brinquedos               0
Gasto com Moveis                   0
Gasto com Utilidades               0
Gasto com Alimentos                0
Gasto com Vestuario                0
Numero de Compras com Desconto     0
Numero de Compras na Web           0
Numero de Compras via Catalogo     0
Numero de Compras na Loja          0
Numero Visitas WebSite Mes         0
Compra na Campanha 1               0
Compra na Campanha 2               0
Compra na Campanha 3               0
Compra na Campanha 4               0
Compra na Campanha 5               0
Comprou                            0
Pais                               0
d

In [8]:
dataset_marketing.duplicated().sum()

np.int64(0)

In [9]:
eda_report(dataset_marketing, nome='Marketing')

EDA Report: Marketing

Colunas: 27
Linhas:  2000

Dtypes:
  str:   4
  int:   22
  float: 1
  data:  0

Non-Null:
  ID: 2000
  Ano Nascimento: 2000
  Escolaridade: 2000
  Estado Civil: 2000
  Salario Anual: 1981
  Filhos em Casa: 2000
  Adolescentes em Casa: 2000
  Data Cadastro: 2000
  Dias Desde Ultima Compra: 2000
  Gasto com Eletronicos: 2000
  Gasto com Brinquedos: 2000
  Gasto com Moveis: 2000
  Gasto com Utilidades: 2000
  Gasto com Alimentos: 2000
  Gasto com Vestuario: 2000
  Numero de Compras com Desconto: 2000
  Numero de Compras na Web: 2000
  Numero de Compras via Catalogo: 2000
  Numero de Compras na Loja: 2000
  Numero Visitas WebSite Mes: 2000
  Compra na Campanha 1: 2000
  Compra na Campanha 2: 2000
  Compra na Campanha 3: 2000
  Compra na Campanha 4: 2000
  Compra na Campanha 5: 2000
  Comprou: 2000
  Pais: 2000

Unique:
  ID: 2000
  Ano Nascimento: 57
  Escolaridade: 5
  Estado Civil: 3
  Salario Anual: 1769
  Filhos em Casa: 3
  Adolescentes em Casa: 3
  Data Cadast

# Limpeza dos Dataset

## Correção dos tipos de dados

## Tratamento de Valores NaN

In [10]:
sao_NA = dataset_marketing['Salario Anual'].isna()
dataset_marketing[sao_NA]

,ID,Ano Nascimento,Escolaridade,Estado Civil,Salario Anual,Filhos em Casa,Adolescentes em Casa,Data Cadastro,Dias Desde Ultima Compra,Gasto com Eletronicos,...,Numero de Compras via Catalogo,Numero de Compras na Loja,Numero Visitas WebSite Mes,Compra na Campanha 1,Compra na Campanha 2,Compra na Campanha 3,Compra na Campanha 4,Compra na Campanha 5,Comprou,Pais
134,8996,1957,Doutorado,Solteiro,NaN,2,1,11/04/2022,4,230,...,2,8,9,0,0,0,0,0,0,Alemanha
262,1994,1983,Curso Superior,Solteiro,NaN,1,0,11/03/2023,11,5,...,0,2,7,0,0,0,0,0,0,Portugal
394,3769,1972,Doutorado,Casado,NaN,1,0,03/02/2018,17,25,...,0,3,7,0,0,0,0,0,0,Brasil
449,5255,1986,Curso Superior,Solteiro,NaN,1,0,02/05/2022,19,5,...,0,0,1,0,0,0,0,0,0,Brasil
525,8268,1961,Doutorado,Solteiro,NaN,0,1,07/11/2020,23,352,...,1,7,6,0,0,0,0,0,0,Chile
590,10629,1973,Segundo Grau,Solteiro,NaN,1,0,9/02/2022,25,25,...,0,3,8,0,0,0,0,0,0,Alemanha
899,10475,1970,Mestrado,Casado,NaN,0,1,04/01/2020,39,187,...,2,6,5,0,0,0,0,0,0,Portugal
997,9235,1957,Curso Superior,Solteiro,NaN,1,1,5/03/2021,45,7,...,0,2,7,0,0,0,0,0,0,Alemanha
1185,7187,1969,Mestrado,Casado,NaN,1,1,05/05/2022,52,375,...,10,4,3,0,0,0,0,0,0,Brasil
1312,8557,1982,Curso Superior,Solteiro,NaN,1,0,6/02/2023,57,11,...,0,3,6,0,0,0,0,0,0,Brasil


In [11]:
dataset_marketing[sao_NA].describe()

,ID,Ano Nascimento,Salario Anual,Filhos em Casa,Adolescentes em Casa,Dias Desde Ultima Compra,Gasto com Eletronicos,Gasto com Brinquedos,Gasto com Moveis,Gasto com Utilidades,...,Numero de Compras na Web,Numero de Compras via Catalogo,Numero de Compras na Loja,Numero Visitas WebSite Mes,Compra na Campanha 1,Compra na Campanha 2,Compra na Campanha 3,Compra na Campanha 4,Compra na Campanha 5,Comprou
count,19.000000,19.000000,0.0,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,...,19.000000,19.000000,19.000000,19.000000,19.0,19.000000,19.000000,19.000000,19.0,19.000000
mean,6206.631579,1968.842105,NaN,0.684211,0.526316,52.842105,218.578947,23.105263,106.526316,30.631579,...,4.315789,1.947368,5.052632,5.157895,0.0,0.105263,0.052632,0.105263,0.0,0.052632
std,3055.614581,12.289395,NaN,0.582393,0.611775,28.998588,263.685064,41.016114,157.377596,45.526806,...,5.812368,2.613505,3.357805,2.566086,0.0,0.315302,0.229416,0.315302,0.0,0.229416
min,1612.000000,1943.000000,NaN,0.000000,0.000000,4.000000,5.000000,0.000000,3.000000,0.000000,...,1.000000,0.000000,0.000000,1.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000
25%,3335.500000,1958.500000,NaN,0.000000,0.000000,24.000000,21.000000,1.000000,14.000000,1.000000,...,1.500000,0.000000,3.000000,3.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000
50%,5798.000000,1970.000000,NaN,1.000000,0.000000,57.000000,81.000000,4.000000,27.000000,3.000000,...,3.000000,1.000000,4.000000,6.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000
75%,8776.500000,1977.000000,NaN,1.000000,1.000000,81.000000,363.500000,28.500000,118.500000,50.500000,...,4.500000,3.000000,7.500000,7.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000
max,10629.000000,1989.000000,NaN,2.000000,2.000000,87.000000,861.000000,138.000000,490.000000,164.000000,...,27.000000,10.000000,12.000000,9.000000,0.0,1.000000,1.000000,1.000000,0.0,1.000000


#### GeekforGeeks -  Regressão

Tipos de Regressão

A regressão pode ser classificada em diferentes tipos com base no número de variáveis ​​preditoras e na natureza da relação entre as variáveis:

1. Regressão Linear Simples

A regressão linear simples modela a relação entre uma variável independente e uma variável dependente contínua ajustando uma reta que minimiza a soma dos quadrados dos erros. Ela pressupõe uma taxa de variação constante, o que significa que a saída varia proporcionalmente à entrada.

* Aplicação: Estimativa do preço de uma casa com base apenas em seu tamanho.
* Vantagem: Altamente interpretável devido à sua estrutura matemática simples.
* Desvantagem: Não consegue capturar padrões de dados curvos ou complexos.

2. Regressão Linear Múltipla

A regressão linear múltipla amplia a regressão linear simples ao incorporar múltiplas variáveis ​​independentes para prever um resultado contínuo. A cada preditor é atribuído um coeficiente que reflete seu impacto individual, mantendo as demais variáveis ​​constantes.

* Aplicação: Previsão de preços de imóveis residenciais utilizando múltiplos fatores como tamanho, localização, idade e número de cômodos.
* Vantagem: Captura a influência combinada de muitos fatores simultaneamente.
* Desvantagem: O desempenho cai na presença de multicolinearidade (características altamente correlacionadas entre si).

3. Regressão Polinomial

A regressão polinomial modela relações não lineares transformando as variáveis ​​de entrada em termos polinomiais de grau superior (por exemplo, x², x³). Embora ajuste curvas não lineares, ela permanece um modelo linear em termos de parâmetros.

* Aplicação: Modelagem de tendências de crescimento curvilíneas, como aumento populacional ou variação de temperatura.
* Vantagem: Captura eficazmente relações não lineares sem recorrer a algoritmos não lineares.
* Desvantagem: Polinômios de grau mais elevado podem levar a sobreajuste e previsões instáveis.

4. Regressão Ridge e Lasso

Ridge e Lasso são técnicas de regressão linear regularizada que adicionam termos de penalização para limitar coeficientes grandes e reduzir o sobreajuste. Ridge (L2) reduz os coeficientes suavemente, enquanto Lasso (L1) pode reduzir alguns coeficientes a zero, permitindo a seleção de características.

* Aplicação: Utilizado em conjuntos de dados de alta dimensionalidade, como atribuição de marketing ou dados de expressão gênica.
* Vantagem: Controla o sobreajuste e melhora a generalização, especialmente com muitos preditores.
* Desvantagem: Os termos de penalização tornam a interpretação do modelo menos direta.

5. Regressão por Vetores de Suporte (SVR)

A Regressão por Vetores de Suporte (SVR) aplica os princípios das Máquinas de Vetores de Suporte (SVMs) a tarefas de regressão. Ela ajusta uma função dentro de uma margem definida (tubo épsilon) e penaliza erros somente quando as previsões ficam fora desse limite. As funções kernel permitem que a SVR modele relações não lineares.

* Aplicação: Previsão de resultados contínuos, como valores de ações ou consumo de energia.
* Vantagem: Funciona bem com conjuntos de dados complexos e de alta dimensionalidade, além de padrões não lineares.
* Desvantagem: Computacionalmente intensivo e requer ajuste cuidadoso dos kernels e parâmetros.

6. Regressão por Árvore de Decisão

A Regressão por Árvore de Decisão divide os dados em ramos hierárquicos com base em limiares de características. Cada nó interno representa uma questão de decisão e os nós folha representam valores contínuos previstos. Ela aprende padrões particionando os dados recursivamente para minimizar erros de previsão.

* Aplicação: Previsão do comportamento de gastos do cliente com base em características demográficas e financeiras.
* Vantagem: Lógica de decisão fácil de visualizar e compreender
* Desvantagem: Facilmente sofre sobreajuste, especialmente quando a árvore se torna profunda e complexa.

7. Regressão por Floresta Aleatória

A Regressão por Floresta Aleatória é um método de conjunto que constrói múltiplas árvores de decisão usando diferentes amostras de dados e calcula a média de suas previsões. Isso reduz a tendência de sobreajuste de árvores individuais e melhora a precisão por meio da diversidade (bagging). Cada árvore captura um aspecto ligeiramente diferente dos dados.

* Aplicação: Previsão de vendas, planejamento de demanda, previsão de rotatividade de clientes.
* Vantagem: Alta precisão e desempenho robusto mesmo em conjuntos de dados ruidosos.
* Desvantagem: Atua como um modelo de caixa preta, dificultando a interpretação devido ao grande número de árvores.

Métricas de avaliação de regressão

Em aprendizado de máquina, a avaliação mede o desempenho de um modelo. Aqui estão algumas métricas de avaliação populares para regressão:

* Erro Médio Absoluto (MAE):  A diferença absoluta média entre os valores previstos e os valores reais da variável alvo.
* Erro Quadrático Médio (EQM):  A diferença quadrática média entre os valores previstos e os valores reais da variável alvo.
* Erro Quadrático Médio (RMSE) :  Raiz quadrada do erro quadrático médio.
* Função de Perda de Huber:  Uma função de perda híbrida que transita de MAE para MSE para erros maiores, proporcionando um equilíbrio entre robustez e a sensibilidade do MSE a valores discrepantes.
* R² – Pontuação : Valores mais altos indicam melhor ajuste, variando de 0 a 1.

#### RandomForestRegressor

In [13]:
'''
dados_sem_na = dataset_marketing[sao_NA == False]

df_previsores = dados_sem_na.drop(columns=dados_sem_na.columns[[0, 4]])
previsores = df_previsores.values
colunas = df_previsores.columns
classe = dados_sem_na.iloc[:, 4].values
# Transformação dos atributos categóricos em atributos numéricos, passando o índice de cada atributo categórico
encoders = {}
lista_colunas_categorias = [1, 2, 5, 24]

for coluna in lista_colunas_categorias:
    le = LabelEncoder()
    previsores[:, coluna] = le.fit_transform(previsores[:, coluna])
    encoders[coluna] = le

n_max_previsores = 10
melhor_r2 = -np.inf
melhor_combo = None
melhor_colunas = None
todos_indices = list(range(previsores.shape[1]))

TEMPO_LIMITE = 3600
inicio_total = time.time()
encerrado_por_tempo = False

print("Testando combinações...\n")
'''

'\ndados_sem_na = dataset_marketing[sao_NA == False]\n\ndf_previsores = dados_sem_na.drop(columns=dados_sem_na.columns[[0, 4]])\nprevisores = df_previsores.values\ncolunas = df_previsores.columns\nclasse = dados_sem_na.iloc[:, 4].values\n# Transformação dos atributos categóricos em atributos numéricos, passando o índice de cada atributo categórico\nencoders = {}\nlista_colunas_categorias = [1, 2, 5, 24]\n\nfor coluna in lista_colunas_categorias:\n    le = LabelEncoder()\n    previsores[:, coluna] = le.fit_transform(previsores[:, coluna])\n    encoders[coluna] = le\n\nn_max_previsores = 10\nmelhor_r2 = -np.inf\nmelhor_combo = None\nmelhor_colunas = None\ntodos_indices = list(range(previsores.shape[1]))\n\nTEMPO_LIMITE = 3600\ninicio_total = time.time()\nencerrado_por_tempo = False\n\nprint("Testando combinações...\n")\n'

In [14]:
'''
%%time
for tamanho in range(1, n_max_previsores + 1):
    if encerrado_por_tempo:
        break

    inicio_tamanho = time.time()
    print(f"--- Tamanho {tamanho} ---")

    for combo in combinations(todos_indices, tamanho):
        tempo_decorrido = time.time() - inicio_total

        if tempo_decorrido >= TEMPO_LIMITE:
            print(f"\n  Limite de {TEMPO_LIMITE}s atingido — finalizando última iteração...")
            encerrado_por_tempo = True
            break

        combo = list(combo)
        X_new = previsores[:, combo]
        Xtr, Xte, ytr, yte = train_test_split(
            X_new, classe, test_size=0.3, random_state=0
        )
        modelo = RandomForestRegressor(
            n_estimators=20,
            random_state=0,
            n_jobs=-1
        )
        modelo.fit(Xtr, ytr)
        r2 = r2_score(yte, modelo.predict(Xte))

        if r2 > melhor_r2:
            melhor_r2 = r2
            melhor_combo = combo
            melhor_colunas = colunas[combo].tolist()
            elapsed = time.time() - inicio_total
            print(f"  Tempo decorrido : {elapsed:.1f}s ({elapsed/60:.1f} min) >>> "
                  f"Novo melhor → R²: {r2:.4f} >>> "
                  f"{len(combo)} features: {melhor_colunas}")

        if melhor_r2 >= 0.80:
            print("\nMeta atingida!")
            encerrado_por_tempo = True
            break

    else:
        tempo_tamanho = time.time() - inicio_tamanho
        print(f"  Tamanho {tamanho} concluído em {tempo_tamanho:.1f}s\n")
        continue
    break

# ── retreino final ────────────────────────────────────────────────────────
print("\nRetreino final com 100 árvores...")
X_best = previsores[:, melhor_combo]
Xtr, Xte, ytr, yte = train_test_split(X_best, classe, test_size=0.3, random_state=0)
modelo_final = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-2)
modelo_final.fit(Xtr, ytr)
r2_final  = r2_score(yte, modelo_final.predict(Xte))
mae_final = mean_absolute_error(yte, modelo_final.predict(Xte))

tempo_total = time.time() - inicio_total
motivo = "Meta de R² atingida" if melhor_r2 >= 0.80 else \
         "Tempo limite atingido" if encerrado_por_tempo else \
         "Todas combinações testadas"

print("\n--- RESULTADO FINAL ---")
print(f"Motivo de parada  : {motivo}")
print(f"Melhor R²         : {r2_final:.4f}")
print(f"MAE               : {mae_final:.2f}")
print(f"Número de features: {len(melhor_combo)}")
print(f"Features usadas   : {melhor_colunas}")
print(f"Tempo total       : {tempo_total:.1f}s  ({tempo_total/60:.1f} min)")
'''

'\n%%time\nfor tamanho in range(1, n_max_previsores + 1):\n    if encerrado_por_tempo:\n        break\n\n    inicio_tamanho = time.time()\n    print(f"--- Tamanho {tamanho} ---")\n\n    for combo in combinations(todos_indices, tamanho):\n        tempo_decorrido = time.time() - inicio_total\n\n        if tempo_decorrido >= TEMPO_LIMITE:\n            print(f"\n  Limite de {TEMPO_LIMITE}s atingido — finalizando última iteração...")\n            encerrado_por_tempo = True\n            break\n\n        combo = list(combo)\n        X_new = previsores[:, combo]\n        Xtr, Xte, ytr, yte = train_test_split(\n            X_new, classe, test_size=0.3, random_state=0\n        )\n        modelo = RandomForestRegressor(\n            n_estimators=20,\n            random_state=0,\n            n_jobs=-1\n        )\n        modelo.fit(Xtr, ytr)\n        r2 = r2_score(yte, modelo.predict(Xte))\n\n        if r2 > melhor_r2:\n            melhor_r2 = r2\n            melhor_combo = combo\n            melh

Testando combinações...

--- Tamanho 1 ---

Tempo decorrido : 0.1s >>> Novo melhor → R²: 0.0009 >>> 1 features: ['Ano Nascimento']    
Tempo decorrido : 0.1s >>> Novo melhor → R²: 0.0144 >>> 1 features: ['Escolaridade']    
Tempo decorrido : 0.2s >>> Novo melhor → R²: 0.0787 >>> 1 features: ['Filhos em Casa']    
Tempo decorrido : 0.4s >>> Novo melhor → R²: 0.1615 >>> 1 features: ['Gasto com Eletronicos']    
Tempo decorrido : 0.5s >>> Novo melhor → R²: 0.1902 >>> 1 features: ['Gasto com Moveis']    

  Tamanho 1 concluído em 1.3s

--- Tamanho 2 ---

Tempo decorrido : 0.5s >>> Novo melhor → R²: 0.1942 >>> 2 features: ['Ano Nascimento', 'Gasto com Moveis']    
Tempo decorrido : 6.0s >>> Novo melhor → R²: 0.2046 >>> 2 features: ['Data Cadastro', 'Gasto com Moveis']    
Tempo decorrido : 6.9s >>> Novo melhor → R²: 0.2225 >>> 2 features: ['Dias Desde Ultima Compra', 'Gasto com Moveis']    
Tempo decorrido : 8.2s >>> Novo melhor → R²: 0.2377 >>> 2 features: ['Gasto com Eletronicos', 'Numero Visitas WebSite Mes']    
Tempo decorrido : 9.8s >>> Novo melhor → R²: 0.2384 >>> 2 features: ['Gasto com Moveis', 'Numero Visitas WebSite Mes']    
Tempo decorrido : 12.9s >>> Novo melhor → R²: 0.2595 >>> 2 features: ['Numero de Compras na Web', 'Numero Visitas WebSite Mes']    

  Tamanho 2 concluído em 15.6s

--- Tamanho 3 ---

Tempo decorrido : 6.8s >>> Novo melhor → R²: 0.2635 >>> 3 features: ['Ano Nascimento', 'Gasto com Eletronicos', 'Numero Visitas WebSite Mes']    
Tempo decorrido : 8.4s >>> Novo melhor → R²: 0.2678 >>> 3 features: ['Ano Nascimento', 'Gasto com Moveis', 'Numero Visitas WebSite Mes']    
Tempo decorrido : 38.1s >>> Novo melhor → R²: 0.2682 >>> 3 features: ['Estado Civil', 'Numero de Compras na Web', 'Numero Visitas WebSite Mes']    
Tempo decorrido : 44.6s >>> Novo melhor → R²: 0.2734 >>> 3 features: ['Filhos em Casa', 'Gasto com Eletronicos', 'Numero Visitas WebSite Mes']    
Tempo decorrido : 72.8s >>> Novo melhor → R²: 0.2743 >>> 3 features: ['Dias Desde Ultima Compra', 'Gasto com Moveis', 'Numero Visitas WebSite Mes']    
Tempo decorrido : 94.9s >>> Novo melhor → R²: 0.2808 >>> 3 features: ['Gasto com Moveis', 'Numero de Compras via Catalogo', 'Numero Visitas WebSite Mes']    

  Tamanho 3 concluído em 120.5s

--- Tamanho 4 ---

Tempo decorrido : 20.7s >>> Novo melhor → R²: 0.2840 >>> 4 features: ['Ano Nascimento', 'Estado Civil', 'Gasto com Alimentos', 'Compra na Campanha 3']    
Tempo decorrido : 48.3s >>> Novo melhor → R²: 0.2947 >>> 4 features: ['Ano Nascimento', 'Data Cadastro', 'Gasto com Moveis', 'Gasto com Alimentos']    
Tempo decorrido : 218.7s >>> Novo melhor → R²: 0.3078 >>> 4 features: ['Estado Civil', 'Data Cadastro', 'Gasto com Moveis', 'Gasto com Alimentos']    

  Tamanho 4 concluído em 650.8s

--- Tamanho 5 ---

Tempo decorrido : 114.8s >>> Novo melhor → R²: 0.3195 >>> 5 features: ['Ano Nascimento', 'Estado Civil', 'Data Cadastro', 'Gasto com Moveis', 'Gasto com Alimentos']    
Tempo decorrido : 1155.2s >>> Novo melhor → R²: 0.3257 >>> 5 features: ['Estado Civil', 'Data Cadastro', 'Gasto com Moveis', 'Gasto com Alimentos', 'Compra na Campanha 3']    
  Tamanho 5 concluído em 2741.3s

--- Tamanho 6 ---

Tempo decorrido : 608.8s >>> Novo melhor → R²: 0.3295 >>> 6 features: ['Ano Nascimento', 'Estado Civil', 'Data Cadastro', 'Gasto com Moveis', 'Gasto com Alimentos', 'Compra na Campanha 3']    

CPU times: total: 1h 53min 58s
Wall time: 1h 17min 52s

#### GradientBoosting

##### CPU

In [15]:
'''
dados_sem_na = dataset_marketing[sao_NA == False]

df_previsores = dados_sem_na.drop(columns=dados_sem_na.columns[[0, 4]])
previsores = df_previsores.values
colunas = df_previsores.columns
classe = dados_sem_na.iloc[:, 4].values

encoders = {}
lista_colunas_categorias = [1, 2, 5, 24]

for coluna in lista_colunas_categorias:
    le = LabelEncoder()
    previsores[:, coluna] = le.fit_transform(previsores[:, coluna])
    encoders[coluna] = le

# ── configurações da busca ────────────────────────────────────────────────
N_MAX_FEATURES = 10
TEMPO_LIMITE   = 3600*8  # segundos

modelos = {
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=100,
        random_state=0
    ),
    "XGBoost": XGBRegressor(
        n_estimators=100,
        random_state=0,
        tree_method="hist",
        device="cpu",
        n_jobs=-1          # ← 7 dos 8 núcleos do 5700G
    ),
}

todos_indices = list(range(previsores.shape[1]))

melhor_r2      = -np.inf
melhor_combo   = None
melhor_colunas = None
melhor_modelo  = None

inicio_total = time.time()
encerrado_por_tempo = False

print("Iniciando busca...\n")
'''

'\ndados_sem_na = dataset_marketing[sao_NA == False]\n\ndf_previsores = dados_sem_na.drop(columns=dados_sem_na.columns[[0, 4]])\nprevisores = df_previsores.values\ncolunas = df_previsores.columns\nclasse = dados_sem_na.iloc[:, 4].values\n\nencoders = {}\nlista_colunas_categorias = [1, 2, 5, 24]\n\nfor coluna in lista_colunas_categorias:\n    le = LabelEncoder()\n    previsores[:, coluna] = le.fit_transform(previsores[:, coluna])\n    encoders[coluna] = le\n\n# ── configurações da busca ────────────────────────────────────────────────\nN_MAX_FEATURES = 10\nTEMPO_LIMITE   = 3600*8  # segundos\n\nmodelos = {\n    "GradientBoosting": GradientBoostingRegressor(\n        n_estimators=100,\n        random_state=0\n    ),\n    "XGBoost": XGBRegressor(\n        n_estimators=100,\n        random_state=0,\n        tree_method="hist",\n        device="cpu",\n        n_jobs=-1          # ← 7 dos 8 núcleos do 5700G\n    ),\n}\n\ntodos_indices = list(range(previsores.shape[1]))\n\nmelhor_r2      = -n

In [16]:
'''
for tamanho in range(1, N_MAX_FEATURES + 1):
    if encerrado_por_tempo:
        break

    inicio_tamanho = time.time()
    total_combos = len(list(combinations(todos_indices, tamanho)))
    print(f"--- Tamanho {tamanho} | {total_combos} combinações ---")

    for combo in combinations(todos_indices, tamanho):
        tempo_decorrido = time.time() - inicio_total

        # verifica tempo — mas deixa a iteração atual terminar
        if tempo_decorrido >= TEMPO_LIMITE:
            print(f"\n  Limite de {TEMPO_LIMITE}s atingido — finalizando última iteração...")
            encerrado_por_tempo = True
            break

        combo = list(combo)
        X_new = previsores[:, combo]
        Xtr, Xte, ytr, yte = train_test_split(
            X_new, classe, test_size=0.3, random_state=0
        )

        for nome, modelo in modelos.items():
            modelo.fit(Xtr, ytr)
            r2  = r2_score(yte, modelo.predict(Xte))
            mae = mean_absolute_error(yte, modelo.predict(Xte))

            if r2 > melhor_r2:
                melhor_r2      = r2
                melhor_combo   = combo
                melhor_colunas = colunas[combo].tolist()
                melhor_modelo  = nome
                elapsed        = time.time() - inicio_total
                print(f"  Novo melhor → R²: {r2:.4f} | MAE: {mae:.2f} | "
                      f"{len(combo)} features | Modelo: {nome}")
                print(f"  Features        : {melhor_colunas}")
                print(f"  Tempo decorrido : {elapsed:.1f}s ({elapsed/60:.1f} min)\n")

    tempo_tamanho = time.time() - inicio_tamanho
    if not encerrado_por_tempo:
        print(f"  Tamanho {tamanho} concluído em {tempo_tamanho:.1f}s\n")

# ── resultado final ───────────────────────────────────────────────────────
tempo_total = time.time() - inicio_total
print("\n" + "="*55)
print("RESULTADO FINAL")
print("="*55)
print(f"Melhor modelo     : {melhor_modelo}")
print(f"Melhor R²         : {melhor_r2:.4f}")
print(f"Número de features: {len(melhor_combo)}")
print(f"Features usadas   : {melhor_colunas}")
print(f"Tempo total       : {tempo_total:.1f}s ({tempo_total/60:.1f} min)")
print("="*55)
'''

'\nfor tamanho in range(1, N_MAX_FEATURES + 1):\n    if encerrado_por_tempo:\n        break\n\n    inicio_tamanho = time.time()\n    total_combos = len(list(combinations(todos_indices, tamanho)))\n    print(f"--- Tamanho {tamanho} | {total_combos} combinações ---")\n\n    for combo in combinations(todos_indices, tamanho):\n        tempo_decorrido = time.time() - inicio_total\n\n        # verifica tempo — mas deixa a iteração atual terminar\n        if tempo_decorrido >= TEMPO_LIMITE:\n            print(f"\n  Limite de {TEMPO_LIMITE}s atingido — finalizando última iteração...")\n            encerrado_por_tempo = True\n            break\n\n        combo = list(combo)\n        X_new = previsores[:, combo]\n        Xtr, Xte, ytr, yte = train_test_split(\n            X_new, classe, test_size=0.3, random_state=0\n        )\n\n        for nome, modelo in modelos.items():\n            modelo.fit(Xtr, ytr)\n            r2  = r2_score(yte, modelo.predict(Xte))\n            mae = mean_absolute_er

--- Tamanho 1 | 25 combinações ---

  Novo melhor → R²: 0.0068 | MAE: 17759.52 | 1 features | Modelo: GradientBoosting
  Features        : ['Ano Nascimento']
  Tempo decorrido : 1.5s (0.0 min)

  Novo melhor → R²: 0.0135 | MAE: 17830.54 | 1 features | Modelo: GradientBoosting
  Features        : ['Escolaridade']
  Tempo decorrido : 1.6s (0.0 min)

  Novo melhor → R²: 0.0777 | MAE: 14393.63 | 1 features | Modelo: GradientBoosting
  Features        : ['Filhos em Casa']
  Tempo decorrido : 1.7s (0.0 min)

  Novo melhor → R²: 0.1933 | MAE: 10024.25 | 1 features | Modelo: GradientBoosting
  Features        : ['Gasto com Eletronicos']
  Tempo decorrido : 2.0s (0.0 min)

  Novo melhor → R²: 0.2071 | MAE: 10140.29 | 1 features | Modelo: GradientBoosting
  Features        : ['Gasto com Moveis']
  Tempo decorrido : 2.2s (0.0 min)

  Tamanho 1 concluído em 1.9s

--- Tamanho 2 | 300 combinações ---

  Novo melhor → R²: 0.2169 | MAE: 9701.21 | 2 features | Modelo: GradientBoosting
  Features        : ['Ano Nascimento', 'Gasto com Moveis']
  Tempo decorrido : 4.2s (0.1 min)

  Novo melhor → R²: 0.2214 | MAE: 8591.27 | 2 features | Modelo: GradientBoosting
  Features        : ['Gasto com Eletronicos', 'Gasto com Moveis']
  Tempo decorrido : 17.0s (0.3 min)

  Novo melhor → R²: 0.2699 | MAE: 8139.68 | 2 features | Modelo: GradientBoosting
  Features        : ['Gasto com Eletronicos', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 17.9s (0.3 min)

  Novo melhor → R²: 0.2727 | MAE: 9108.60 | 2 features | Modelo: GradientBoosting
  Features        : ['Gasto com Moveis', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 21.0s (0.4 min)

  Tamanho 2 concluído em 27.0s

--- Tamanho 3 | 2300 combinações ---

  Novo melhor → R²: 0.2842 | MAE: 7969.07 | 3 features | Modelo: GradientBoosting
  Features        : ['Ano Nascimento', 'Gasto com Eletronicos', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 45.0s (0.7 min)

  Novo melhor → R²: 0.2905 | MAE: 7877.41 | 3 features | Modelo: GradientBoosting
  Features        : ['Filhos em Casa', 'Gasto com Eletronicos', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 114.9s (1.9 min)

  Novo melhor → R²: 0.2913 | MAE: 7815.57 | 3 features | Modelo: GradientBoosting
  Features        : ['Gasto com Eletronicos', 'Gasto com Alimentos', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 190.7s (3.2 min)

  Tamanho 3 concluído em 237.1s

--- Tamanho 4 | 12650 combinações ---

  Novo melhor → R²: 0.2934 | MAE: 12509.70 | 4 features | Modelo: XGBoost
  Features        : ['Ano Nascimento', 'Estado Civil', 'Adolescentes em Casa', 'Gasto com Alimentos']
  Tempo decorrido : 300.1s (5.0 min)

  Novo melhor → R²: 0.3023 | MAE: 13087.15 | 4 features | Modelo: XGBoost
  Features        : ['Ano Nascimento', 'Estado Civil', 'Gasto com Alimentos', 'Compra na Campanha 4']
  Tempo decorrido : 315.1s (5.3 min)

  Tamanho 4 concluído em 1453.9s

--- Tamanho 5 | 53130 combinações ---

  Novo melhor → R²: 0.3314 | MAE: 11597.50 | 5 features | Modelo: XGBoost
  Features        : ['Ano Nascimento', 'Estado Civil', 'Adolescentes em Casa', 'Gasto com Alimentos', 'Compra na Campanha 3']
  Tempo decorrido : 1985.6s (33.1 min)

  Tamanho 5 concluído em 6689.7s

--- Tamanho 6 | 177100 combinações ---

  Novo melhor → R²: 0.3340 | MAE: 11365.98 | 6 features | Modelo: XGBoost
  Features        : ['Ano Nascimento', 'Estado Civil', 'Adolescentes em Casa', 'Gasto com Alimentos', 'Compra na Campanha 2', 'Compra na Campanha 3']
  Tempo decorrido : 9921.6s (165.4 min)


  Limite de 28800s atingido — finalizando última iteração...

=======================================================

RESULTADO FINAL

=======================================================

Melhor modelo     : XGBoost
Melhor R²         : 0.3340
Número de features: 6
Features usadas   : ['Ano Nascimento', 'Estado Civil', 'Adolescentes em Casa', 'Gasto com Alimentos', 'Compra na Campanha 2', 'Compra na Campanha 3']
Tempo total       : 28800.1s (480.0 min)

=======================================================

##### GPU

In [17]:
'''
dados_sem_na = dataset_marketing[sao_NA == False]

df_previsores = dados_sem_na.drop(columns=dados_sem_na.columns[[0, 4]])
previsores = df_previsores.values
colunas = df_previsores.columns
classe = dados_sem_na.iloc[:, 4].values

encoders = {}
lista_colunas_categorias = [1, 2, 5, 24]

for coluna in lista_colunas_categorias:
    le = LabelEncoder()
    previsores[:, coluna] = le.fit_transform(previsores[:, coluna])
    encoders[coluna] = le

# ── configurações da busca ────────────────────────────────────────────────
N_MAX_FEATURES = 5
TEMPO_LIMITE = 3600

modelos = {
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=0),
    "XGBoost": XGBRegressor(
        n_estimators=100, random_state=0, device="cuda", tree_method="hist"
    ),
}

todos_indices = list(range(previsores.shape[1]))

melhor_r2 = -np.inf
melhor_combo = None
melhor_colunas = None
melhor_modelo = None

inicio_total = time.time()
encerrado_por_tempo = False

print("Iniciando busca...\n")
'''

'\ndados_sem_na = dataset_marketing[sao_NA == False]\n\ndf_previsores = dados_sem_na.drop(columns=dados_sem_na.columns[[0, 4]])\nprevisores = df_previsores.values\ncolunas = df_previsores.columns\nclasse = dados_sem_na.iloc[:, 4].values\n\nencoders = {}\nlista_colunas_categorias = [1, 2, 5, 24]\n\nfor coluna in lista_colunas_categorias:\n    le = LabelEncoder()\n    previsores[:, coluna] = le.fit_transform(previsores[:, coluna])\n    encoders[coluna] = le\n\n# ── configurações da busca ────────────────────────────────────────────────\nN_MAX_FEATURES = 5\nTEMPO_LIMITE = 3600\n\nmodelos = {\n    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=0),\n    "XGBoost": XGBRegressor(\n        n_estimators=100, random_state=0, device="cuda", tree_method="hist"\n    ),\n}\n\ntodos_indices = list(range(previsores.shape[1]))\n\nmelhor_r2 = -np.inf\nmelhor_combo = None\nmelhor_colunas = None\nmelhor_modelo = None\n\ninicio_total = time.time()\nencerrado_por_tempo = False\

In [18]:
'''
for tamanho in range(1, N_MAX_FEATURES + 1):
    if encerrado_por_tempo:
        break

    inicio_tamanho = time.time()
    total_combos = len(list(combinations(todos_indices, tamanho)))
    print(f"--- Tamanho {tamanho} | {total_combos} combinações ---")

    for combo in combinations(todos_indices, tamanho):
        tempo_decorrido = time.time() - inicio_total

        if tempo_decorrido >= TEMPO_LIMITE:
            print(f"\n  Limite de {TEMPO_LIMITE}s atingido — finalizando última iteração...")
            encerrado_por_tempo = True
            break

        combo = list(combo)
        X_new = previsores[:, combo]
        Xtr, Xte, ytr, yte = train_test_split(
            X_new, classe, test_size=0.3, random_state=0
        )

        for nome, modelo in modelos.items():
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                modelo.fit(Xtr, ytr)
                previsoes = modelo.predict(Xte)

            r2 = r2_score(yte, previsoes)
            mae = mean_absolute_error(yte, previsoes)

            if r2 > melhor_r2:
                melhor_r2 = r2
                melhor_combo = combo
                melhor_colunas = colunas[combo].tolist()
                melhor_modelo = nome
                elapsed = time.time() - inicio_total
                print(
                    f"  Novo melhor → R²: {r2:.4f} | MAE: {mae:.2f} | "
                    f"{len(combo)} features | Modelo: {nome}"
                )
                print(f"  Features        : {melhor_colunas}")
                print(f"  Tempo decorrido : {elapsed:.1f}s ({elapsed/60:.1f} min)\n")

    tempo_tamanho = time.time() - inicio_tamanho
    if not encerrado_por_tempo:
        print(f"  Tamanho {tamanho} concluído em {tempo_tamanho:.1f}s\n")

# ── resultado final ───────────────────────────────────────────────────────
tempo_total = time.time() - inicio_total
print("\n" + "=" * 55)
print("RESULTADO FINAL")
print("=" * 55)
print(f"Melhor modelo     : {melhor_modelo}")
print(f"Melhor R²         : {melhor_r2:.4f}")
print(f"Número de features: {len(melhor_combo)}")
print(f"Features usadas   : {melhor_colunas}")
print(f"Tempo total       : {tempo_total:.1f}s ({tempo_total/60:.1f} min)")
print("=" * 55)
'''

'\nfor tamanho in range(1, N_MAX_FEATURES + 1):\n    if encerrado_por_tempo:\n        break\n\n    inicio_tamanho = time.time()\n    total_combos = len(list(combinations(todos_indices, tamanho)))\n    print(f"--- Tamanho {tamanho} | {total_combos} combinações ---")\n\n    for combo in combinations(todos_indices, tamanho):\n        tempo_decorrido = time.time() - inicio_total\n\n        if tempo_decorrido >= TEMPO_LIMITE:\n            print(f"\n  Limite de {TEMPO_LIMITE}s atingido — finalizando última iteração...")\n            encerrado_por_tempo = True\n            break\n\n        combo = list(combo)\n        X_new = previsores[:, combo]\n        Xtr, Xte, ytr, yte = train_test_split(\n            X_new, classe, test_size=0.3, random_state=0\n        )\n\n        for nome, modelo in modelos.items():\n            with warnings.catch_warnings():\n                warnings.simplefilter("ignore")\n                modelo.fit(Xtr, ytr)\n                previsoes = modelo.predict(Xte)\n\n   

--- Tamanho 1 | 25 combinações ---

  Novo melhor → R²: 0.0068 | MAE: 17759.52 | 1 features | Modelo: GradientBoosting
  Features        : ['Ano Nascimento']
  Tempo decorrido : 1.1s (0.0 min)

  Novo melhor → R²: 0.0135 | MAE: 17830.54 | 1 features | Modelo: GradientBoosting
  Features        : ['Escolaridade']
  Tempo decorrido : 1.5s (0.0 min)

  Novo melhor → R²: 0.0777 | MAE: 14393.63 | 1 features | Modelo: GradientBoosting
  Features        : ['Filhos em Casa']
  Tempo decorrido : 1.9s (0.0 min)

  Novo melhor → R²: 0.1933 | MAE: 10024.25 | 1 features | Modelo: GradientBoosting
  Features        : ['Gasto com Eletronicos']
  Tempo decorrido : 2.8s (0.0 min)

  Novo melhor → R²: 0.2071 | MAE: 10140.29 | 1 features | Modelo: GradientBoosting
  Features        : ['Gasto com Moveis']
  Tempo decorrido : 3.5s (0.1 min)

  Tamanho 1 concluído em 6.1s

--- Tamanho 2 | 300 combinações ---

  Novo melhor → R²: 0.2169 | MAE: 9701.21 | 2 features | Modelo: GradientBoosting
  Features        : ['Ano Nascimento', 'Gasto com Moveis']
  Tempo decorrido : 10.1s (0.2 min)

  Novo melhor → R²: 0.2214 | MAE: 8591.27 | 2 features | Modelo: GradientBoosting
  Features        : ['Gasto com Eletronicos', 'Gasto com Moveis']
  Tempo decorrido : 56.6s (0.9 min)

  Novo melhor → R²: 0.2699 | MAE: 8139.68 | 2 features | Modelo: GradientBoosting
  Features        : ['Gasto com Eletronicos', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 59.9s (1.0 min)

  Novo melhor → R²: 0.2727 | MAE: 9108.60 | 2 features | Modelo: GradientBoosting
  Features        : ['Gasto com Moveis', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 71.3s (1.2 min)

  Tamanho 2 concluído em 99.8s

--- Tamanho 3 | 2300 combinações ---

  Novo melhor → R²: 0.2842 | MAE: 7969.07 | 3 features | Modelo: GradientBoosting
  Features        : ['Ano Nascimento', 'Gasto com Eletronicos', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 157.8s (2.6 min)

  Novo melhor → R²: 0.2905 | MAE: 7877.41 | 3 features | Modelo: GradientBoosting
  Features        : ['Filhos em Casa', 'Gasto com Eletronicos', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 414.6s (6.9 min)

  Novo melhor → R²: 0.2913 | MAE: 7815.57 | 3 features | Modelo: GradientBoosting
  Features        : ['Gasto com Eletronicos', 'Gasto com Alimentos', 'Numero Visitas WebSite Mes']
  Tempo decorrido : 686.3s (11.4 min)

  Tamanho 3 concluído em 885.3s

-----------------------------------------------------------------------------------------

### Outros

Regressão Linear Simples 

Regressão Linear Múltipla

Regressão Polinomial

Regressão Ridge e Lasso

Regressão por Vetores de Suporte (SVR)

Regressão por Árvore de Decisão

In [15]:
dados_sem_na = dataset_marketing[sao_NA == False]
# ── Simples usa só a feature mais importante ──────────────────────────────
X_simples = dados_sem_na[["Gasto com Alimentos"]].values

df_previsores = dados_sem_na.drop(columns=dados_sem_na.columns[[0, 4]])
previsores = df_previsores.values
colunas = df_previsores.columns
classe = dados_sem_na.iloc[:, 4].values

encoders = {}
lista_colunas_categorias = [1, 2, 5, 24]
for coluna in lista_colunas_categorias:
    le = LabelEncoder()
    previsores[:, coluna] = le.fit_transform(previsores[:, coluna])
    encoders[coluna] = le

Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(
    X_simples, classe, test_size=0.3, random_state=0
)

Xtr_all, Xte_all, ytr_all, yte_all = train_test_split(
    previsores, classe, test_size=0.3, random_state=0
)
floresta_all = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-2)
floresta_all.fit(Xtr_all, ytr_all)

importancias = pd.Series(
    floresta_all.feature_importances_, index=colunas
).sort_values(ascending=False)
# ── top 10 features ───────────────────────────────────────────────────────
idx_top10 = [list(colunas).index(c) for c in importancias.head(10).index]
X_top10   = previsores[:, idx_top10]

# simples usa só a feature mais importante
idx_top1  = [idx_top10[0]]
X_simples = previsores[:, idx_top1]

Xtr_10, Xte_10, ytr_10, yte_10 = train_test_split(
    X_top10, classe, test_size=0.3, random_state=0
)
Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(
    X_simples, classe, test_size=0.3, random_state=0
)

modelos = {
    "Linear Simples":     ("simples",  LinearRegression()),
    "Linear Múltipla":    ("multipla", LinearRegression()),
    "Polinomial":         ("multipla", Pipeline([
                                ("poly", PolynomialFeatures(degree=2)),
                                ("reg",  LinearRegression())
                             ])),
    "Ridge":              ("multipla", Ridge(alpha=1.0)),
    "Lasso":              ("multipla", Lasso(alpha=1.0)),
    "SVR":                ("multipla", Pipeline([
                                ("scaler", StandardScaler()),
                                ("svr",    SVR(kernel="rbf"))
                             ])),
    "Árvore de Decisão":  ("multipla", DecisionTreeRegressor(random_state=0)),
    "Random Forest":      ("multipla", RandomForestRegressor(
                                n_estimators=100, random_state=0, n_jobs=-1)),
    "Gradient Boosting":  ("multipla", GradientBoostingRegressor(
                                n_estimators=100, random_state=0)),
    "XGBoost":           ("multipla", XGBRegressor(
                                n_estimators=100, random_state=0,
                                tree_method="hist", device="cpu", n_jobs=-1)),
}

resultados = {}
for nome, (tipo, modelo) in modelos.items():
    inicio = time.time()

    if tipo == "simples":
        modelo.fit(Xtr_s, ytr_s)
        previsoes = modelo.predict(Xte_s)
        y_ref = yte_s
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            modelo.fit(Xtr_10, ytr_10)
            previsoes = modelo.predict(Xte_10)
        y_ref = yte_10

    r2  = r2_score(y_ref, previsoes)
    mae = mean_absolute_error(y_ref, previsoes)
    tempo = time.time() - inicio
    resultados[nome] = r2
    print(f"{nome:<25} {r2:>8.4f} {mae:>14.2f} {tempo:>7.1f}s")

resultados_ordenados = dict(sorted(resultados.items(), key=lambda x: x[1], reverse=True))

melhor_nome = list(resultados_ordenados.keys())[0]
melhor_r2   = list(resultados_ordenados.values())[0]

print("\n" + "=" * 58)
print(f"{'Posição':<10} {'Modelo':<25} {'R²':>8}")
print("-" * 58)
for i, (nome, r2) in enumerate(resultados_ordenados.items(), 1):
    destaque = " ← melhor" if i == 1 else ""
    print(f"{i:<10} {nome:<25} {r2:>8.4f}{destaque}")

print("=" * 58)
print(f"\nMelhor: {melhor_nome} → R²: {melhor_r2:.4f}")

Linear Simples              0.1540       12274.50     0.0s
Linear Múltipla             0.2586        8285.52     0.0s
Polinomial                  0.2597        8015.15     0.0s
Ridge                       0.2587        8285.44     0.0s
Lasso                       0.2587        8285.50     0.0s
SVR                         0.0040       18072.96     0.1s
Árvore de Decisão           0.1699        9113.78     0.0s
Random Forest               0.2559        6952.98     0.2s
Gradient Boosting           0.2793        7018.87     0.2s
XGBoost                     0.2819        6723.03     0.2s

Posição    Modelo                          R²
----------------------------------------------------------
1          XGBoost                     0.2819 ← melhor
2          Gradient Boosting           0.2793
3          Polinomial                  0.2597
4          Ridge                       0.2587
5          Lasso                       0.2587
6          Linear Múltipla             0.2586
7          Random F

**Regra geral do R² mínimo:**

| R² | Interpretação |
|---|---|
| < 0.30 | Fraco — features não explicam o target |
| 0.30 – 0.50 | Razoável — alguma relação mas muito ruído |
| 0.50 – 0.70 | Bom |
| 0.70 – 0.90 | Muito bom |
| > 0.90 | Excelente (ou overfitting) |


#### PolynomialFeatures

In [16]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

dados_sem_na = dataset_marketing[sao_NA == False]
# ── Simples usa só a feature mais importante ──────────────────────────────
X_simples = dados_sem_na[["Gasto com Alimentos"]].values

df_previsores = dados_sem_na.drop(columns=dados_sem_na.columns[[0, 4]])
previsores = df_previsores.values
colunas = df_previsores.columns
classe = dados_sem_na.iloc[:, 4].values

Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(
    X_simples, classe, test_size=0.3, random_state=0
)

encoders = {}
lista_colunas_categorias = [1, 2, 5, 24]
for coluna in lista_colunas_categorias:
    le = LabelEncoder()
    previsores[:, coluna] = le.fit_transform(previsores[:, coluna])
    encoders[coluna] = le

idx_top10 = [list(colunas).index(c) for c in importancias.head(10).index]
X_top10   = previsores[:, idx_top10]

Xtr, Xte, ytr, yte = train_test_split(
    X_top10, classe, test_size=0.3, random_state=0
)

print(f"Features usadas: {importancias.head(10).index.tolist()}\n")
print(f"{'Modelo':<30} {'R²':>8} {'MAE':>14} {'Features geradas':>18} {'Tempo':>8}")
print("-" * 80)

resultados_poly = {}
for grau in [1, 2, 3, 4]:
    inicio = time.time()

    poly = PolynomialFeatures(degree=grau)
    # só para contar quantas features são geradas
    n_features = poly.fit_transform(Xtr).shape[1]

    modelo = Pipeline([
        ("poly",   PolynomialFeatures(degree=grau)),
        ("scaler", StandardScaler()),          # ← escala ajuda muito no polinomial
        ("reg",    LinearRegression())
    ])

    modelo.fit(Xtr, ytr)
    previsoes = modelo.predict(Xte)

    r2  = r2_score(yte, previsoes)
    mae = mean_absolute_error(yte, previsoes)
    tempo = time.time() - inicio
    nome = f"Polinomial grau {grau}"
    resultados_poly[nome] = r2

    print(f"{nome:<30} {r2:>8.4f} {mae:>14.2f} {n_features:>18} {tempo:>7.1f}s")

print("\n" + "=" * 80)
melhor = max(resultados_poly, key=resultados_poly.get)
print(f"Melhor: {melhor} → R²: {resultados_poly[melhor]:.4f}")
print(f"Baseline XGBoost → R²: 0.3340")

Features usadas: ['Gasto com Eletronicos', 'Gasto com Moveis', 'Numero de Compras via Catalogo', 'Numero de Compras com Desconto', 'Numero Visitas WebSite Mes', 'Numero de Compras na Loja', 'Adolescentes em Casa', 'Gasto com Alimentos', 'Data Cadastro', 'Numero de Compras na Web']

Modelo                               R²            MAE   Features geradas    Tempo
--------------------------------------------------------------------------------
Polinomial grau 1                0.2586        8285.52                 11     0.0s
Polinomial grau 2                0.2597        8015.15                 66     0.0s
Polinomial grau 3                0.2592        7980.90                286     0.1s
Polinomial grau 4              -163.6072      130514.31               1001     0.3s

Melhor: Polinomial grau 2 → R²: 0.2597
Baseline XGBoost → R²: 0.3340


In [17]:
import time
from itertools import combinations
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# ── conta combinações ─────────────────────────────────────────────────────
print(f"{'Tamanho':<10} {'Combinações':>12}")
print("-" * 24)
total = 0
for t in range(1, 11):
    n = len(list(combinations(range(10), t)))
    total += n
    print(f"{t:<10} {n:>12}")
print(f"{'TOTAL':<10} {total:>12}")

# ── mede tempo médio por grau com 1 combinação de cada tamanho ───────────
print("\nTempo estimado por grau (1 combinação):\n")
print(f"{'Grau':<8} {'Features':>10} {'Tempo/fit':>12} {'Total estimado':>16}")
print("-" * 48)

combo_teste = list(range(5))  # 5 features como amostra
X_teste_fit = previsores[:, combo_teste]
Xtr_t, Xte_t, ytr_t, yte_t = train_test_split(
    X_teste_fit, classe, test_size=0.3, random_state=0
)

for grau in [1, 2, 3, 4]:
    poly = PolynomialFeatures(degree=grau)
    n_feat = poly.fit_transform(Xtr_t).shape[1]

    modelo = Pipeline([
        ("poly",   PolynomialFeatures(degree=grau)),
        ("scaler", StandardScaler()),
        ("reg",    LinearRegression())
    ])

    inicio = time.time()
    modelo.fit(Xtr_t, ytr_t)
    modelo.predict(Xte_t)
    tempo_fit = time.time() - inicio

    total_estimado = tempo_fit * total  # 1023 combinações
    print(f"{grau:<8} {n_feat:>10} {tempo_fit:>11.3f}s {total_estimado:>14.1f}s "
          f"({total_estimado/60:.1f} min)")

Tamanho     Combinações
------------------------
1                    10
2                    45
3                   120
4                   210
5                   252
6                   210
7                   120
8                    45
9                    10
10                    1
TOTAL              1023

Tempo estimado por grau (1 combinação):

Grau       Features    Tempo/fit   Total estimado
------------------------------------------------
1                 6       0.003s            2.8s (0.0 min)
2                21       0.004s            4.2s (0.1 min)
3                56       0.007s            7.6s (0.1 min)
4               126       0.028s           28.7s (0.5 min)


In [18]:
print(f"Features usadas: {importancias.head(10).index.tolist()}\n")
print(f"{'Modelo':<30} {'R²':>8} {'MAE':>14} {'Features geradas':>18} {'Tempo':>8}")
print("-" * 80)

resultados_poly = {}

for grau in [1, 2, 3, 4]:
    for tamanho in range(1, 11):
        for combo in combinations(range(10), tamanho):
            combo = list(combo)
            X_combo = previsores[:, [idx_top10[i] for i in combo]]
            Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
                X_combo, classe, test_size=0.3, random_state=0
            )

            poly    = PolynomialFeatures(degree=grau)
            n_feat  = poly.fit_transform(Xtr_c).shape[1]

            modelo  = Pipeline([
                ("poly",   PolynomialFeatures(degree=grau)),
                ("scaler", StandardScaler()),
                ("reg",    LinearRegression())
            ])

            inicio = time.time()
            modelo.fit(Xtr_c, ytr_c)
            previsoes = modelo.predict(Xte_c)
            tempo = time.time() - inicio

            r2  = r2_score(yte_c, previsoes)
            mae = mean_absolute_error(yte_c, previsoes)
            nome = f"Grau {grau} | {tamanho} features"

            if nome not in resultados_poly or r2 > resultados_poly[nome][0]:
                resultados_poly[nome] = (r2, mae, n_feat, tempo,
                                         colunas[[idx_top10[i] for i in combo]].tolist())

# ── resultado ordenado ────────────────────────────────────────────────────
resultados_ordenados = sorted(resultados_poly.items(), key=lambda x: x[1][0], reverse=True)

print(f"\n{'Posição':<8} {'Modelo':<25} {'R²':>8} {'MAE':>14} {'Features':>18}")
print("-" * 80)
for i, (nome, (r2, mae, n_feat, tempo, feats)) in enumerate(resultados_ordenados[:15], 1):
    destaque = " ← melhor" if i == 1 else ""
    print(f"{i:<8} {nome:<25} {r2:>8.4f} {mae:>14.2f} {n_feat:>18}{destaque}")
    if i == 1:
        print(f"         Features: {feats}")

print("\n" + "=" * 80)
melhor_nome, (melhor_r2, melhor_mae, _, _, melhor_feats) = resultados_ordenados[0]
print(f"Melhor modelo   : {melhor_nome}")
print(f"Melhor R²       : {melhor_r2:.4f}")
print(f"MAE             : {melhor_mae:.2f}")
print(f"Features usadas : {melhor_feats}")
print(f"Baseline XGBoost: R² 0.3340")

Features usadas: ['Gasto com Eletronicos', 'Gasto com Moveis', 'Numero de Compras via Catalogo', 'Numero de Compras com Desconto', 'Numero Visitas WebSite Mes', 'Numero de Compras na Loja', 'Adolescentes em Casa', 'Gasto com Alimentos', 'Data Cadastro', 'Numero de Compras na Web']

Modelo                               R²            MAE   Features geradas    Tempo
--------------------------------------------------------------------------------

Posição  Modelo                          R²            MAE           Features
--------------------------------------------------------------------------------
1        Grau 4 | 4 features         0.3006        7996.37                 70 ← melhor
         Features: ['Gasto com Moveis', 'Numero Visitas WebSite Mes', 'Gasto com Alimentos', 'Numero de Compras na Web']
2        Grau 3 | 6 features         0.2983        7996.50                 84
3        Grau 3 | 5 features         0.2982        8067.76                 56
4        Grau 4 | 3 features 

### Final - Usar a média pois os ML estão abaixo da média

In [12]:
dataset_marketing.iloc[:, 4] = dataset_marketing.iloc[:, 4].fillna(
    dataset_marketing.iloc[:, 4].mean())
dataset_marketing.iloc[:,4] = dataset_marketing.iloc[:,4].round(2)

In [13]:
dataset_marketing.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 27 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ID                              2000 non-null   int64  
 1   Ano Nascimento                  2000 non-null   int64  
 2   Escolaridade                    2000 non-null   str    
 3   Estado Civil                    2000 non-null   str    
 4   Salario Anual                   2000 non-null   float64
 5   Filhos em Casa                  2000 non-null   int64  
 6   Adolescentes em Casa            2000 non-null   int64  
 7   Data Cadastro                   2000 non-null   str    
 8   Dias Desde Ultima Compra        2000 non-null   int64  
 9   Gasto com Eletronicos           2000 non-null   int64  
 10  Gasto com Brinquedos            2000 non-null   int64  
 11  Gasto com Moveis                2000 non-null   int64  
 12  Gasto com Utilidades            2000 non-null

In [14]:
dataset_marketing.iloc[130:136,4].head()

130    72071.00
131    59235.00
132    21994.00
133    21994.00
134    52290.85
Name: Salario Anual, dtype: float64

## Tratamento de Valores Duplicados

## Tratamento de Dados Outliers

In [15]:
desvio_padrão = dataset_marketing.iloc[:, 4].std()
print('DP: ',desvio_padrão.round(2),'Media: ',dataset_marketing.iloc[:,4].mean().round(2))
print(dataset_marketing.iloc[:,4].mean().round(2)+3*desvio_padrão.round(2))
dataset_marketing[dataset_marketing.iloc[:,4] > dataset_marketing.iloc[:,4].mean()+3*desvio_padrão]

DP:  25363.3 Media:  52290.85
128380.75


,ID,Ano Nascimento,Escolaridade,Estado Civil,Salario Anual,Filhos em Casa,Adolescentes em Casa,Data Cadastro,Dias Desde Ultima Compra,Gasto com Eletronicos,...,Numero de Compras via Catalogo,Numero de Compras na Loja,Numero Visitas WebSite Mes,Compra na Campanha 1,Compra na Campanha 2,Compra na Campanha 3,Compra na Campanha 4,Compra na Campanha 5,Comprou,Pais
325,4931,1977,Curso Superior,Casado,157146.0,0,0,4/07/2023,13,1,...,28,0,1,0,0,0,0,0,0,Espanha
497,1501,1982,Doutorado,Solteiro,160803.0,0,0,08/04/2019,21,55,...,28,1,0,0,0,0,0,0,0,Portugal
527,9432,1977,Curso Superior,Casado,666666.0,1,0,06/02/2020,23,9,...,1,3,6,0,0,0,0,0,0,Espanha
731,1503,1976,Doutorado,Casado,162397.0,1,1,06/03/2020,31,85,...,0,1,1,0,0,0,0,0,0,Estados Unidos
853,5336,1971,Mestrado,Casado,157733.0,1,0,06/04/2020,37,39,...,0,1,1,0,0,0,0,0,0,Estados Unidos
1826,5555,1975,Curso Superior,Divorciado,153924.0,0,0,02/07/2018,81,1,...,0,0,0,0,0,0,0,0,0,Estados Unidos
1925,11181,1949,Doutorado,Solteiro,156924.0,0,0,8/07/2023,85,2,...,0,0,0,0,0,0,0,0,0,Chile


In [16]:
desvio_padrao = dataset_marketing.iloc[:, 4].std()
media = dataset_marketing.loc[:,'Salario Anual'].mean()
media = media.round(2)
print('DP: ',desvio_padrao.round(2),'Media: ',dataset_marketing.iloc[:,4].mean().round(2))
print(dataset_marketing.iloc[:,4].mean().round(2)+5*desvio_padrao.round(2))
dataset_marketing.loc[dataset_marketing['Salario Anual'] > media + 5 * desvio_padrao, 'Salario Anual']

DP:  25363.3 Media:  52290.85
179107.35


527    666666.0
Name: Salario Anual, dtype: float64

In [17]:
dataset_marketing.loc[dataset_marketing['Salario Anual'] > media + 5 * desvio_padrao, 'Salario Anual'] = media

In [18]:
dataset_marketing.loc[dataset_marketing['Salario Anual'] > media + 5 * desvio_padrao, 'Salario Anual']

Series([], Name: Salario Anual, dtype: float64)

In [19]:
dataset_marketing.describe()

,ID,Ano Nascimento,Salario Anual,Filhos em Casa,Adolescentes em Casa,Dias Desde Ultima Compra,Gasto com Eletronicos,Gasto com Brinquedos,Gasto com Moveis,Gasto com Utilidades,...,Numero de Compras na Web,Numero de Compras via Catalogo,Numero de Compras na Loja,Numero Visitas WebSite Mes,Compra na Campanha 1,Compra na Campanha 2,Compra na Campanha 3,Compra na Campanha 4,Compra na Campanha 5,Comprou
count,2000.000000,2000.000000,2000.00000,2000.000000,2000.000000,2000.00000,2000.00000,2000.000000,2000.000000,2000.000000,...,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,5617.382500,1968.797000,51983.66500,0.447500,0.503000,43.73500,303.92850,26.316500,164.143500,37.587000,...,4.075000,2.635500,5.797500,5.327500,0.074000,0.073000,0.071500,0.066500,0.013000,0.160000
std,3259.910118,11.981468,21316.18507,0.535151,0.540497,25.85885,337.84483,40.317925,221.565768,54.748143,...,2.754663,2.885793,3.275952,2.440947,0.261836,0.260202,0.257723,0.249216,0.113302,0.366698
min,0.000000,1893.000000,1730.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2814.750000,1959.000000,35409.00000,0.000000,0.000000,22.00000,23.00000,1.000000,16.000000,3.000000,...,2.000000,0.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5492.000000,1970.000000,52199.00000,0.000000,0.000000,45.00000,175.50000,8.000000,67.000000,12.000000,...,4.000000,2.000000,5.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,8495.000000,1977.000000,68117.25000,1.000000,1.000000,66.00000,503.25000,32.000000,226.000000,50.000000,...,6.000000,4.000000,8.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,11191.000000,1996.000000,162397.00000,2.000000,2.000000,88.00000,1493.00000,199.000000,1725.000000,259.000000,...,27.000000,28.000000,13.000000,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [20]:
media_ano_nascimento = dataset_marketing['Ano Nascimento'].mean()
desvio_padrao_ano_nascimento = dataset_marketing['Ano Nascimento'].std()
limite_inferior_ano_nascimento = media_ano_nascimento - 3*desvio_padrao_ano_nascimento
limite_superior_ano_nascimento = media_ano_nascimento + 3*desvio_padrao_ano_nascimento
print(f'Limite data: {limite_inferior_ano_nascimento.round(0)}  |---------| {limite_superior_ano_nascimento.round(0)}\nMedia data: {(media_ano_nascimento).round(0)}')

Limite data: 1933.0  |---------| 2005.0
Media data: 1969.0


In [21]:
dataset_marketing.loc[
    dataset_marketing['Ano Nascimento'] < limite_inferior_ano_nascimento, 
    'Ano Nascimento'
] = dataset_marketing.loc[
    dataset_marketing['Ano Nascimento'] < limite_inferior_ano_nascimento, 
    'Ano Nascimento'
] + 100

In [22]:
# 513 e 827 > 1893 1899

dataset_marketing.loc[[513, 827],'Ano Nascimento']

513    1993
827    1999
Name: Ano Nascimento, dtype: int64

# Criando Novo arquivo csv para gerar o dashboard

In [23]:
dataset_marketing.to_csv(f'dados_marketing/dados_marketing_clear.csv', index=False, encoding='utf-8')

In [25]:
colunas_campanhas = [
    'Compra na Campanha 1',
    'Compra na Campanha 2',
    'Compra na Campanha 3',
    'Compra na Campanha 4',
    'Compra na Campanha 5',
]

df_long = dataset_marketing.melt(
    value_vars=colunas_campanhas,
    var_name='Campanha',
    value_name='Valor'
)

df_long['Status'] = df_long['Valor'].map({1: 'Comprou', 0: 'Não Comprou'})
df_long = df_long.drop(columns='Valor')

print(df_long.head(10))
df_long.to_csv('campanhas_long.csv', index=False)

               Campanha       Status
0  Compra na Campanha 1  Não Comprou
1  Compra na Campanha 1  Não Comprou
2  Compra na Campanha 1  Não Comprou
3  Compra na Campanha 1  Não Comprou
4  Compra na Campanha 1  Não Comprou
5  Compra na Campanha 1  Não Comprou
6  Compra na Campanha 1  Não Comprou
7  Compra na Campanha 1  Não Comprou
8  Compra na Campanha 1  Não Comprou
9  Compra na Campanha 1  Não Comprou


In [25]:
dataset_marketing[dataset_marketing['Comprou']==1].count()

ID                                320
Ano Nascimento                    320
Escolaridade                      320
Estado Civil                      320
Salario Anual                     320
Filhos em Casa                    320
Adolescentes em Casa              320
Data Cadastro                     320
Dias Desde Ultima Compra          320
Gasto com Eletronicos             320
Gasto com Brinquedos              320
Gasto com Moveis                  320
Gasto com Utilidades              320
Gasto com Alimentos               320
Gasto com Vestuario               320
Numero de Compras com Desconto    320
Numero de Compras na Web          320
Numero de Compras via Catalogo    320
Numero de Compras na Loja         320
Numero Visitas WebSite Mes        320
Compra na Campanha 1              320
Compra na Campanha 2              320
Compra na Campanha 3              320
Compra na Campanha 4              320
Compra na Campanha 5              320
Comprou                           320
Pais        

In [30]:
dataset_marketing[(dataset_marketing['Comprou'] == 1)  & (dataset_marketing['Compra na Campanha 1'] == 1)].count()

ID                                74
Ano Nascimento                    74
Escolaridade                      74
Estado Civil                      74
Salario Anual                     74
Filhos em Casa                    74
Adolescentes em Casa              74
Data Cadastro                     74
Dias Desde Ultima Compra          74
Gasto com Eletronicos             74
Gasto com Brinquedos              74
Gasto com Moveis                  74
Gasto com Utilidades              74
Gasto com Alimentos               74
Gasto com Vestuario               74
Numero de Compras com Desconto    74
Numero de Compras na Web          74
Numero de Compras via Catalogo    74
Numero de Compras na Loja         74
Numero Visitas WebSite Mes        74
Compra na Campanha 1              74
Compra na Campanha 2              74
Compra na Campanha 3              74
Compra na Campanha 4              74
Compra na Campanha 5              74
Comprou                           74
Pais                              74
d

In [31]:
dataset_marketing[dataset_marketing['Compra na Campanha 1'] == 1].count()

ID                                148
Ano Nascimento                    148
Escolaridade                      148
Estado Civil                      148
Salario Anual                     148
Filhos em Casa                    148
Adolescentes em Casa              148
Data Cadastro                     148
Dias Desde Ultima Compra          148
Gasto com Eletronicos             148
Gasto com Brinquedos              148
Gasto com Moveis                  148
Gasto com Utilidades              148
Gasto com Alimentos               148
Gasto com Vestuario               148
Numero de Compras com Desconto    148
Numero de Compras na Web          148
Numero de Compras via Catalogo    148
Numero de Compras na Loja         148
Numero Visitas WebSite Mes        148
Compra na Campanha 1              148
Compra na Campanha 2              148
Compra na Campanha 3              148
Compra na Campanha 4              148
Compra na Campanha 5              148
Comprou                           148
Pais        